In [2]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import $ivy.`io.delta::delta-spark_4.1:4.2.0`

import $ivy.$
import $ivy.$

In [3]:
//spark session 
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("ProyectoBigDataSilver")
  .master("local[*]")
  .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
  .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/11 20:04:21 WARN Utils: Your hostname, SamuraiLoco, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/11 20:04:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/05/11 20:04:21 INFO SparkContext: Running Spark version 4.1.1
26/05/11 20:04:21 INFO SparkContext: OS info Linux, 6.6.114.1-microsoft-standard-WSL2, amd64
26/05/11 20:04:21 INFO SparkContext: Java version 17.0.18+8-Ubuntu-124.04.1
26/05/11 20:04:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/11 20:04:22 INFO ResourceUtils: ==============================================================
26/05/11 20:04:22 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/11 20:04:22 INFO ResourceUtils: ==============================================================
26/05/11 20:04:22

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@3110fd5f

In [4]:
//rutas 
val basePath = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA"

val bronzeOrdersPath = basePath + "/delta/bronze/orders"
val bronzeMarketingPath = basePath + "/delta/bronze/marketing_campaigns"

val silverPath = basePath + "/delta/silver/orders_clean"

basePath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA"
bronzeOrdersPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/delta/bronze/orders"
bronzeMarketingPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/delta/bronze/marketing_campaigns"
silverPath: String = "/mnt/c/Users/danie/OneDrive/Desktop/Proyectos/Proyecto_final_BIGDATA/delta/silver/orders_clean"

In [5]:
//leer bronze 
val ordersBronzeDF = spark.read
  .format("delta")
  .load(bronzeOrdersPath)

val marketingBronzeDF = spark.read
  .format("delta")
  .load(bronzeMarketingPath)

println(s"Orders Bronze: ${ordersBronzeDF.count()} filas")
println(s"Marketing Bronze: ${marketingBronzeDF.count()} filas")

Orders Bronze: 10000 filas
Marketing Bronze: 8 filas


ordersBronzeDF: org.apache.spark.sql.package.DataFrame = [Order_ID: string, Order_Date: date ... 26 more fields]
marketingBronzeDF: org.apache.spark.sql.package.DataFrame = [campaign_cost: string, category: string ... 4 more fields]

In [6]:
//normalizar columnas 
val ordersRenamedDF = ordersBronzeDF
  .withColumnRenamed("Order_ID", "order_id")
  .withColumnRenamed("Order_Date", "order_date")
  .withColumnRenamed("Year", "year")
  .withColumnRenamed("Month", "month")
  .withColumnRenamed("Quarter", "quarter")
  .withColumnRenamed("Season", "season")
  .withColumnRenamed("Customer_ID", "customer_id")
  .withColumnRenamed("Customer_Gender", "customer_gender")
  .withColumnRenamed("Customer_Segment", "customer_segment")
  .withColumnRenamed("Region", "region")
  .withColumnRenamed("Country", "country")
  .withColumnRenamed("Category", "category")
  .withColumnRenamed("Sub_Category", "sub_category")
  .withColumnRenamed("Product_Name", "product_name")
  .withColumnRenamed("Unit_Price", "unit_price")
  .withColumnRenamed("Quantity", "quantity")
  .withColumnRenamed("Discount", "discount")
  .withColumnRenamed("Revenue", "revenue")
  .withColumnRenamed("Cost", "cost")
  .withColumnRenamed("Profit", "profit")
  .withColumnRenamed("Profit_Margin_%", "profit_margin_pct")
  .withColumnRenamed("Shipping_Cost", "shipping_cost")
  .withColumnRenamed("Shipping_Method", "shipping_method")
  .withColumnRenamed("Shipping_Days", "shipping_days")
  .withColumnRenamed("Payment_Method", "payment_method")
  .withColumnRenamed("Order_Status", "order_status")

ordersRenamedDF: org.apache.spark.sql.package.DataFrame = [order_id: string, order_date: date ... 26 more fields]

In [7]:
//limpiar 
val ordersBeforeDedup = ordersRenamedDF.count()

val ordersDedupDF = ordersRenamedDF.dropDuplicates("order_id")

val ordersAfterDedup = ordersDedupDF.count()

println(s"Duplicados eliminados en orders: ${ordersBeforeDedup - ordersAfterDedup}")

Duplicados eliminados en orders: 0


ordersBeforeDedup: Long = 10000L
ordersDedupDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [order_id: string, order_date: date ... 26 more fields]
ordersAfterDedup: Long = 10000L

In [8]:
//corregir nulos y tipos 
val ordersCleanDF = ordersDedupDF
  .filter(col("order_id").isNotNull)
  .filter(col("order_date").isNotNull)
  .filter(col("category").isNotNull)
  .filter(col("country").isNotNull)
  .filter(col("revenue").isNotNull && col("revenue") >= 0)
  .filter(col("profit").isNotNull)
  .withColumn("quantity", coalesce(col("quantity"), lit(1)))
  .withColumn("discount", coalesce(col("discount"), lit(0.0)))
  .withColumn("shipping_cost", coalesce(col("shipping_cost"), lit(0.0)))
  .withColumn("shipping_days", coalesce(col("shipping_days"), lit(0)))

ordersCleanDF: org.apache.spark.sql.package.DataFrame = [order_id: string, order_date: date ... 26 more fields]

In [9]:
//limpiar marketing 
val marketingCleanDF = marketingBronzeDF
  .dropDuplicates()
  .filter(col("category").isNotNull)
  .filter(col("marketing_channel").isNotNull)
  .withColumn("campaign_cost_clean", col("campaign_cost").cast("double"))
  .filter(col("campaign_cost_clean").isNotNull)
  .drop("campaign_cost")
  .withColumnRenamed("campaign_cost_clean", "campaign_cost")
  .groupBy("category")
  .agg(
    first("marketing_channel").alias("marketing_channel"),
    avg("campaign_cost").alias("campaign_cost"),
    first("target_country").alias("target_country")
  )

marketingCleanDF: org.apache.spark.sql.package.DataFrame = [category: string, marketing_channel: string ... 2 more fields]

In [10]:
//join entre fuentes 
val silverDF = ordersCleanDF
  .join(
    marketingCleanDF.select(
      col("category"),
      col("marketing_channel"),
      col("campaign_cost"),
      col("target_country")
    ),
    Seq("category"),
    "left"
  )

silverDF: org.apache.spark.sql.package.DataFrame = [category: string, order_id: string ... 29 more fields]

In [11]:
//verificamos silver 
println(s"Orders antes de limpieza: ${ordersBronzeDF.count()}")
println(s"Orders después de limpieza: ${ordersCleanDF.count()}")
println(s"Marketing antes de limpieza: ${marketingBronzeDF.count()}")
println(s"Marketing después de limpieza: ${marketingCleanDF.count()}")
println(s"Silver final: ${silverDF.count()} filas")

silverDF.printSchema()
silverDF.show(5)

Orders antes de limpieza: 10000
Orders después de limpieza: 10000
Marketing antes de limpieza: 8
Marketing después de limpieza: 5
Silver final: 10000 filas
root
 |-- category: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: string (nullable = true)
 |-- season: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- region: string (nullable = true)
 |-- country: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = false)
 |-- discount: double (nullable = false)
 |-- revenue: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- profit: double (nullable = true)
 |-- profit_margin_pct: do

In [12]:
//guardar silver en delta 
silverDF.write
  .format("delta")
  .mode("overwrite")
  .save(silverPath)

In [13]:
//verificamos 
val silverCheckDF = spark.read
  .format("delta")
  .load(silverPath)

println(s"Silver guardado correctamente con ${silverCheckDF.count()} filas")
silverCheckDF.show(5)

Silver guardado correctamente con 10000 filas
+-------------+----------+----------+----+-----+-------+------+-----------+---------------+----------------+-------------+-------+------------+--------------------+----------+--------+--------+-------+-------+-------+-----------------+-------------+---------------+-------------+--------------+------------+--------------------+--------------+-----------------+-------------+--------------+
|     category|  order_id|order_date|year|month|quarter|season|customer_id|customer_gender|customer_segment|       region|country|sub_category|        product_name|unit_price|quantity|discount|revenue|   cost| profit|profit_margin_pct|shipping_cost|shipping_method|shipping_days|payment_method|order_status| ingestion_timestamp|   source_file|marketing_channel|campaign_cost|target_country|
+-------------+----------+----------+----+-----+-------+------+-----------+---------------+----------------+-------------+-------+------------+--------------------+--------

silverCheckDF: org.apache.spark.sql.package.DataFrame = [category: string, order_id: string ... 29 more fields]